# Significancia estadística con OLS

In [16]:
# Cargar librerías

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [17]:
# Configuración

ruta_archivo = "Base_final_narino_cundinamarca.csv"
df = pd.read_csv(ruta_archivo, encoding="utf-8-sig")

In [18]:
# Definir variable objetivo y variables candidatas

target = "Rendimiento"
columna_tiempo = "Año"

variables_numericas = [
    'Precipitación acumulada anual (mm/año)',
    'Temperatura media anual (°C)',
    'Máximo de la temperatura media mensual (°C)',
    'Mínimo de la temperatura media mensual (°C)',
    'Humedad relativa media anual (%)',
    'Radiación solar acumulada anual (MJ/m²/año)',
    'Humedad volumétrica media anual del suelo capa 1 (m³/m³)',
    'Humedad volumétrica media anual del suelo capa 2 (m³/m³)',
    'Evaporación potencial acumulada anual (mm/año)',
    'SPI1_floracion',
    'SPI1_llenado',
    'altitud_media_m',
    'Rendimiento_lag1',
    'Rendimiento_rolling3',
]

variables_categoricas = [
    "Departamento",
    "Municipio"
]
variables_modelo = variables_numericas + variables_categoricas

In [19]:
# Crear base para OLS

columnas_ols = variables_modelo + [target, columna_tiempo]

df_ols = df[columnas_ols].copy()

for col in variables_numericas + [target]:
    df_ols[col] = pd.to_numeric(df_ols[col], errors="coerce")

for col in variables_categoricas:
    df_ols[col] = df_ols[col].astype("category")

df_ols = df_ols.dropna()

df_ols = df_ols.sort_values(columna_tiempo).reset_index(drop=True)

print("Tamaño base OLS:", df_ols.shape)


Tamaño base OLS: (1481, 18)


In [20]:
# Split temporal
anio_corte = int(df_ols[columna_tiempo].quantile(0.80))

df_ols_train = df_ols[df_ols[columna_tiempo] <= anio_corte].copy()
df_ols_test = df_ols[df_ols[columna_tiempo] > anio_corte].copy()

print("Año de corte:", anio_corte)
print("Train:", df_ols_train.shape)
print("Test:", df_ols_test.shape)

Año de corte: 2021
Train: (1212, 18)
Test: (269, 18)


In [21]:
# Crear variables dummy para Departamento y Municipio
X_train_ols = pd.get_dummies(
    df_ols_train[variables_modelo],
    columns=variables_categoricas,
    drop_first=True
)

X_test_ols = pd.get_dummies(
    df_ols_test[variables_modelo],
    columns=variables_categoricas,
    drop_first=True
)

y_train_ols = df_ols_train[target].astype(float)
y_test_ols = df_ols_test[target].astype(float)

X_train_ols, X_test_ols = X_train_ols.align(
    X_test_ols,
    join="left",
    axis=1,
    fill_value=0
)

X_train_ols = X_train_ols.astype(float)
X_test_ols = X_test_ols.astype(float)

X_train_ols = sm.add_constant(X_train_ols)
X_test_ols = sm.add_constant(X_test_ols, has_constant="add")

In [22]:
# Entrenar modelo OLS

modelo_ols = sm.OLS(y_train_ols, X_train_ols).fit()

print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:            Rendimiento   R-squared:                       0.504
Model:                            OLS   Adj. R-squared:                  0.458
Method:                 Least Squares   F-statistic:                     10.84
Date:                Sun, 03 May 2026   Prob (F-statistic):          6.50e-110
Time:                        16:26:19   Log-Likelihood:                 254.73
No. Observations:                1212   AIC:                            -299.5
Df Residuals:                    1107   BIC:                             236.1
Df Model:                         104                                         
Covariance Type:            nonrobust                                         
                                                               coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [23]:
# Tabla de significancia estadística

tabla_significancia = pd.DataFrame({
    "variable": modelo_ols.params.index,
    "coeficiente": modelo_ols.params.values,
    "p_value": modelo_ols.pvalues.values
})

tabla_significancia = tabla_significancia[
    tabla_significancia["variable"] != "const"
].copy()

tabla_significancia["significativa_10%"] = tabla_significancia["p_value"] < 0.10
tabla_significancia["significativa_5%"] = tabla_significancia["p_value"] < 0.05
tabla_significancia["significativa_1%"] = tabla_significancia["p_value"] < 0.01

tabla_significancia = tabla_significancia.sort_values(
    "p_value"
).reset_index(drop=True)

tabla_significancia.round(6)

,variable,coeficiente,p_value,significativa_10%,significativa_5%,significativa_1%
0,Rendimiento_lag1,0.468799,0.000000,True,True,True
1,altitud_media_m,0.001025,0.000000,True,True,True
2,Municipio_ZIPACON,-0.594559,0.000000,True,True,True
3,Municipio_PACHO,0.959342,0.000000,True,True,True
4,Municipio_CUMBITARA,0.637077,0.000000,True,True,True
...,...,...,...,...,...,...
101,Municipio_BELTRAN,-0.012349,0.940512,False,False,False
102,Municipio_SAN JUAN DE RIOSECO,-0.008922,0.941886,False,False,False
103,Municipio_PASTO,0.006648,0.945888,False,False,False
104,Municipio_RICAURTE,-0.005713,0.961638,False,False,False


In [24]:

tabla_significancia_numericas = tabla_significancia[
    tabla_significancia["variable"].isin(variables_numericas)
].copy()

tabla_significancia_numericas.round(6)

,variable,coeficiente,p_value,significativa_10%,significativa_5%,significativa_1%
0,Rendimiento_lag1,0.468799,0.000000,True,True,True
1,altitud_media_m,0.001025,0.000000,True,True,True
16,Temperatura media anual (°C),0.165417,0.000121,True,True,True
21,Humedad volumétrica media anual del suelo capa...,-13.211362,0.000637,True,True,True
23,Humedad volumétrica media anual del suelo capa...,11.447121,0.000987,True,True,True
29,Radiación solar acumulada anual (MJ/m²/año),-0.000194,0.003163,True,True,True
44,Mínimo de la temperatura media mensual (°C),0.063232,0.032486,True,True,False
55,Máximo de la temperatura media mensual (°C),-0.032833,0.073968,True,False,False
59,Evaporación potencial acumulada anual (mm/año),-0.000088,0.113510,False,False,False
61,Precipitación acumulada anual (mm/año),-0.000026,0.124122,False,False,False


In [25]:
# 10. Evaluar OLS en test

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_train_ols = modelo_ols.predict(X_train_ols)
y_pred_test_ols = modelo_ols.predict(X_test_ols)

rmse_train_ols = mean_squared_error(y_train_ols, y_pred_train_ols) ** 0.5
rmse_test_ols = mean_squared_error(y_test_ols, y_pred_test_ols) ** 0.5

mae_train_ols = mean_absolute_error(y_train_ols, y_pred_train_ols)
mae_test_ols = mean_absolute_error(y_test_ols, y_pred_test_ols)

r2_train_ols = r2_score(y_train_ols, y_pred_train_ols)
r2_test_ols = r2_score(y_test_ols, y_pred_test_ols)

print("RMSE train OLS:", rmse_train_ols)
print("RMSE test OLS:", rmse_test_ols)
print("MAE train OLS:", mae_train_ols)
print("MAE test OLS:", mae_test_ols)
print("R2 train OLS:", r2_train_ols)
print("R2 test OLS:", r2_test_ols)

RMSE train OLS: 0.19610448289987067
RMSE test OLS: 0.3017483339336456
MAE train OLS: 0.1414017050766284
MAE test OLS: 0.22613972504073931
R2 train OLS: 0.5044975720047629
R2 test OLS: 0.211579207820225


In [26]:
# Tabla final

tabla_justificacion_ols = tabla_significancia_numericas.copy()

tabla_justificacion_ols["decision"] = np.where(
    tabla_justificacion_ols["p_value"] < 0.10,
    "Seleccionada por significancia estadística",
    "Revisar con criterio técnico"
)

tabla_justificacion_ols = tabla_justificacion_ols.sort_values(
    "p_value"
).reset_index(drop=True)

tabla_justificacion_ols.round(6)

,variable,coeficiente,p_value,significativa_10%,significativa_5%,significativa_1%,decision
0,Rendimiento_lag1,0.468799,0.000000,True,True,True,Seleccionada por significancia estadística
1,altitud_media_m,0.001025,0.000000,True,True,True,Seleccionada por significancia estadística
2,Temperatura media anual (°C),0.165417,0.000121,True,True,True,Seleccionada por significancia estadística
3,Humedad volumétrica media anual del suelo capa...,-13.211362,0.000637,True,True,True,Seleccionada por significancia estadística
4,Humedad volumétrica media anual del suelo capa...,11.447121,0.000987,True,True,True,Seleccionada por significancia estadística
5,Radiación solar acumulada anual (MJ/m²/año),-0.000194,0.003163,True,True,True,Seleccionada por significancia estadística
6,Mínimo de la temperatura media mensual (°C),0.063232,0.032486,True,True,False,Seleccionada por significancia estadística
7,Máximo de la temperatura media mensual (°C),-0.032833,0.073968,True,False,False,Seleccionada por significancia estadística
8,Evaporación potencial acumulada anual (mm/año),-0.000088,0.113510,False,False,False,Revisar con criterio técnico
9,Precipitación acumulada anual (mm/año),-0.000026,0.124122,False,False,False,Revisar con criterio técnico
